# Exploring the wiki places QA set

An audit of the wiki places QA dataset (`/data/llms4eu/wiki/qa/`), done as if another company had handed it to us:
what is in it, how it was made, what the generation skewed, and whether the labels can be trusted. The notebook is
meant to be readable on its own: section 0 gives all the background you need, so you do not have to read anything
else first.

Run it with `uv sync --group notebook` and the `.venv` kernel. Everything is SQL (DuckDB) straight on the
Parquet files, so each cell takes seconds. Nothing is written to `/data`.

Sections: 0 background · summary · 1 inventory · 2 corpus · 3 popularity bias · 4 difficulty · 5 ambiguity ·
6 what the filters removed · 7 cross-lingual · 8 the other files · 9 generator artifacts · 10 data hygiene ·
11 read examples · 12 hand check · 13 your own questions vs generated · 14 what to measure ·
15 generating more realistic questions.

Each section starts with **why** we look at it and ends with **so what**: the takeaway, what to take into account
when using the data, and what the section does not show yet. The numbers in the text are from the 2026-10-05 build.

## 0. Background: what you need to know to read this notebook

### The project

**LLMs4EU** is an EU project building a democratic LLM system that helps people find the tourist attractions they are
looking for, often nature-based (castles, lakes, caves, parks), in all EU languages. **Datacation's part** is
helping with the fine-tuning. **Right now** we explore *indexing* for retrieval-augmented generation (RAG): how
to split pages into chunks, embed them and search them so that, for a user's question, the right page comes back.
An LLM then writes the answer from the retrieved text. That second step comes later.

To compare indexing methods we need a **test set**: questions where we know which page holds the answer. This
dataset is that test set. It was generated with LLMs by a team member between 25 September and
5 October 2026. Free generation credits run until **10 October**, so anything missing must be generated before then.

### The data in one picture

- **The corpus** (`/data/llms4eu/wiki/pages.jsonl`): 133,164 Wikipedia pages, one per place. Places of 12 types
  (castle, castle ruin, fortification, national park, nature reserve, natural monument, forest, cave, waterfall,
  lake, mountain, garden) in the 27 EU countries were taken from Wikidata. Each place's article is in its country's
  language (a Dutch castle has a Dutch page, a Belgian one Dutch or French), stored as Markdown text. This is what a
  system indexes.
- **The questions** (`/data/llms4eu/wiki/qa/wiki_qa_<file>.parquet`), each pointing at the page(s) that answer it:

| file | what a row is | tests | usable rows |
|---|---|---|---|
| `rag` | a question about one place, with answer and evidence; easy or hard | finding the right page | 498,550 |
| `unanswerable` | a question about a real place that no page answers | saying "I don't know" | 29,315 |
| `compare` | a question about two places ("which is higher?") | combining two pages | 8,519 |
| `meta` | a question whose answer is a list of places (all X in a country, or within N km) | returning a set | 3,892 |
| `tables` | a question that needs a calculation over a table in a page | table handling | 166 |
| `clean`, `challenge` | the raw generated items before filtering (easy and hard) | provenance only | - |

- **Easy** questions (`kind = 'corpus'`) name the place: *"How high is the Dent de Moirans?"*. **Hard** questions
  (`kind = 'challenge'`) describe it without naming it: *"Which castle in the Rhön was probably founded before
  1150?"*. For hard questions the generator was forbidden to use the page title and the page's 25 most
  distinctive words, so that keyword search cannot simply match them.

### How the questions were made

1. **Page tags:** a small model (Gemma 4 E4B, fine-tuned with LoRA, a cheap way to fine-tune by training a small
   add-on) tagged every page, for example `tourist_appeal` low / medium / high.
2. **Easy questions:** up to 5 per page (1-2 for short pages). Most were written by a small local model (Qwen3.5-4B,
   fine-tuned with LoRA to imitate a bigger "teacher"), the rest by the teacher itself, **Space Bunny**.
3. **Quality labels and repairs:** an LLM labelled every question (fluent, realistic, unique, supported...) and
   repaired or dropped the bad ones. Questions that are identical across pages were flagged.
4. **Hard questions:** written by two free preview LLMs, **Space Bunny** (OpenRouter) and **Ling 3.1 Flash**
   (Vercel). Each page went to one of them by a hash of its id (`crc32(id) % 2`: 0 = Bunny, 1 = Ling).
5. **Answers, evidence and translations:** Ling wrote an answer for every question, copied 1-3 sentences from the
   page as evidence, and translated question and answer into one other EU language.
6. **Answer judges:** Ling and Bunny each judged every answer, seeing only the title and the evidence sentences.
   `answer_ok` = every check passed.
7. **Extra files:** Ling generated and checked `unanswerable`, `compare` and `meta` (the `meta` gold lists come from
   Wikidata metadata, Ling only phrased the question). For `tables`, code computed the answers and Sonnet checked them.
8. **Multi-page relevance:** for most hard questions, Ling judged which *other* pages in the keyword-search top 20 fit
   the question too (`relevant`, `hits`).
9. **Spot checks:** Claude Sonnet checked samples of 30-200 items per step blind (without seeing the earlier
   labels). No human checked the data.

So nearly every label is an LLM's opinion, mostly from two free preview models checking each other.

### Terms used throughout

| term | meaning |
|---|---|
| gold page | the page a question was written from: the correct retrieval result |
| hit@k | the gold page is among the top k results (hit@1, hit@10) |
| MRR | mean reciprocal rank: 1 / rank of the gold page, averaged over questions |
| BM25 | classic keyword search; the baseline every method must beat |
| dense retrieval, embedding | search by meaning: texts and questions become vectors, the nearest vectors are returned |
| chunk | a piece of a page; indexing usually embeds chunks, not whole pages |
| LLM judge | an LLM asked to grade an output (is the answer supported? is the translation faithful?) |
| `ok`, `answer_ok` | the row passed every check; only use these rows |
| `split` | `dev` (20%, tune on it) or `test` (80%, report on it), by a hash of the place, so a place is in one split only |
| `balanced` | a subset of at most 400 pages per language, so Swedish does not dominate |
| `lang`, `x_lang`, `question_x` | language of the page and question; the language of the translated question |
| `sitelinks` | in how many Wikipedia languages the place has an article: our measure of how well-known a place is |
| `stub`, `machine_generated` | page shorter than 2,000 characters; page written by a bot (the Swedish lake register) |
| `lex_overlap` | share of the question's (weighted) words that appear in the page, 0-1: low = hard for keyword search |
| `title_in_question` | the place's name appears in the question |
| `rank_o`, `rank_x` | precomputed rank of the gold page for the original / translated question: [dense rank, dense margin, BM25 rank]; 101 = not in the top 100 |
| `evidence`, `spans` | the supporting sentences and their character positions in the page text |
| `hits`, `relevant` | how many pages fit the question (`unique`, `few`, `many`) and which other ones; count them as correct too |
| `pool_saturated` | many candidates matched, so probably more fitting pages exist than were judged |
| `qtype` | answer type: identify (which place?), number, date, location, name, description, reason |
| `variants` | rewrites of the question: keywords only, with typos, verbose, as a follow-up in a conversation |
| infobox | the fact table at the top of a Wikipedia page (height, area, coordinates) |
| abstention | the system says it does not know instead of answering |

Language codes: sv Swedish, de German, fi Finnish, it Italian, fr French, pl Polish, es Spanish, sk Slovak,
cs Czech, lt Lithuanian, nl Dutch, et Estonian, hu Hungarian, ro Romanian, lv Latvian, da Danish, sl Slovenian,
pt Portuguese, bg Bulgarian, el Greek, en English, hr Croatian, ga Irish, lb Luxembourgish, mt Maltese, tr Turkish.

### Sources this notebook refers to

- **"the report"**: `docs/reports/wiki-qgen/wiki-qa-dataset.md` in this repo, how the dataset was made.
- **"the README"**, with its section numbers (§): `/data/llms4eu/wiki/qa/README.md`, how to use each file.
- **"the starter guide"**: the team's one-page introduction to the dataset for new members.
- **SynthTRIPs**: a published synthetic travel-recommendation dataset whose generator favoured popular places even
  when asked for obscure ones. It is the reason we check for popularity bias.
- **"the audit plan"**: audit the dataset first, then use the remaining credits only for what it lacks. The
  candidates were (1) recommendation-style questions with preferences ("a quiet castle near X, good with kids"),
  (2) cross-border geo questions, and (3) a subset balanced between famous and obscure places.

You do not need to open any of these: their relevant content is summarised above, and the "Confirmed" table below
says which findings they already contain.

## Summary: what we learned

What is **new** is not in the dataset docs (`docs/reports/wiki-qgen/`, `/data/llms4eu/wiki/qa/README.md`) or
the starter guide. The rest confirms what they say, which is worth knowing too.

**New**

| finding | evidence | so what |
|---|---|---|
| Hard questions are riddles about the place's identity | 71% of hard questions are `identify` ("which castle ...?") vs 5% of easy ones; the answer is mostly just the name | the hard set tests "find the place from a description". Real users know the name and want a fact, or want a recommendation; neither is tested by the hard set, and answer generation on it is trivial |
| Questions copy the page (examples: end of section 11) | **measured:** hard questions keep 72% of their weighted words in the page (easy 77%); BM25 finds 66% (75% outside sv) without the name; the generator saw the page while writing. **Observed** (hand check): over-detailed, riddle-like questions. **Inferred:** real users, who have not read the page, ask vaguer and in other words | fine for comparing methods *against each other*; probably optimistic as an absolute estimate of performance on real questions. Section 13 turns the inference into a measurement with a small human-written set |
| Hand check (Dutch, n = 21, preliminary): answers right, hard questions unnatural (examples in English: end of section 11) | answer correct 18/18 judged (2 unsure); points to one place 18/19. Easy questions natural 9/9, fluent 8/9. **Hard questions natural 4/10, fluent 5/9**: riddles ("if you know this, you also know the name"), stiff or wrong Dutch (*wordt ... gaf*, *een vertrouwde man*, *een negentiende-eeuwse boswachtershuis die*), and banned words that bend facts (a lake as a *natuurgebied*) | the labels are trustworthy; the problem of the hard set is realism, not correctness. ~10 items per group: the hard vs easy contrast is clear, exact rates are not |
| The hard set is mostly one model | outside Swedish, 91% of hard questions come from Ling (Bunny kept fewer: 76% vs 85%) | a method tuned on hard questions is tuned to Ling's phrasing; check results per generator (section 9) |
| Unanswerable has a shortcut | 31% of not-covered questions are "how much does ..." (prices); a few openings cover most of them | a system can learn "price question -> I don't know" without reading; measure abstention per template |
| Obscure places are harder | hard questions outside sv: BM25 hit@10 56% for one-language places vs 82-87% for better-known ones | report by `sitelinks` bucket; an average hides the long tail a recommender should surface |
| No popularity bias in coverage | 80-97% of pages get questions in every `sitelinks` bucket; only one-language Swedish lake stubs are skipped (29%) | scores are not inflated by famous places |
| Page text is full of wiki boilerplate | edit links (`[redigera \| redigera wikitext]`) on 97% of pages, 3.6% of all chars; tables are 22% of chars (39% sv) | clean pages before chunking and embedding; decide how tables are chunked |
| The headline number depends on the subset | hard BM25 hit@10: 66% on all rows, 81% on `balanced` | always say which subset; compare methods on the same one |
| The dense ranks in the file are a skewed subset | for hard questions 95% of rows with a dense rank are Swedish | do not use `rank_o[1]`; run our own dense baseline |
| Geo questions rarely cross borders | 2-7% of geo questions have gold places in two countries | cross-border questions are missing for the recommender goal |
| The answer filter hit rich pages more | `answer_ok` 81-84% for hu, el and 88-90% for castles, parks vs 95% for lakes | the kept set leans slightly towards simple pages |
| Some duplicates slipped through | 11 questions are asked word for word about several pages (one about 5 Swedish lakes) | small, but these can never be answered uniquely: drop them |
| Category and language are entangled | lakes are sv/fi, castles de/fr/es, mountains de/it/sk | a per-category result is mostly a per-language result |

**Confirmed (already documented)**

| finding | where it is documented |
|---|---|
| Swedish dominates (52% of pages, 39% of questions); report per language or on `balanced` | report, README, starter guide |
| Easy questions are keyword-solvable (BM25 hit@10 ~97%); hard ones are not | report, README §1-2 |
| Lead-only dense fails on hard questions because answers sit in the body (about half after 1,000 chars) | README §2 |
| ~23% of Swedish hard questions fit several lake stubs; count `relevant` pages as hits | report, README §8 |
| BM25 collapses on translated questions; translations have errors in ~1 in 7 passing rows | report, README §3 |
| Meta gold lists are complete for the index, not the real world | README §7 |
| `tables` is too small (166 ok) | report, manifest |
| Labels come from free models checked by Sonnet on samples of 30-200; no human check yet | report. Our hand check (section 12) is new |

In [ ]:
import json
import re
import textwrap
import zlib
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd

QA = Path("/data/llms4eu/wiki/qa")
PAGES = QA.parent / "pages.jsonl"
LANG = "nl"  # language for the examples; questions come with an English gloss
SEED = 7  # change for other examples

pd.set_option("display.max_colwidth", 120, "display.width", 200)
plt.rcParams.update(
    {"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False}
)

con = duckdb.connect()
# which free model wrote a hard question: Bunny on crc32(id) % 2 == 0, Ling on 1 (report, stage 6)
con.create_function(
    "generator",
    lambda i: "bunny" if zlib.crc32(i.encode()) % 2 == 0 else "ling",
    ["VARCHAR"],
    "VARCHAR",
)
for f in ["rag", "unanswerable", "compare", "meta", "tables", "clean", "challenge"]:
    con.sql(f"create view {f} as select * from '{QA}/wiki_qa_{f}.parquet'")
con.sql(f"create table pages as select * exclude (text) from read_json_auto('{PAGES}')")
# rag has no English question: take it from the source items (easy: clean, repaired or not; hard: challenge)
con.sql("""create table gloss as select id, q, any_value(q_en) q_en from (
    select id, item->>'question' q, item->>'question_en' q_en from clean
    union all select id, repaired->>'question', repaired->>'question_en' from clean where starts_with(repaired, '{')
    union all select id, item->>'question', item->>'question_en' from challenge) group by all""")


def q(sql):
    return con.sql(sql).df()


def texts(ids):
    # page id -> text, for the few pages an example needs
    return dict(
        con.execute(
            f"select id, text from read_json_auto('{PAGES}') where list_contains(?, id)",
            [list(ids)],
        ).fetchall()
    )


def wrap(s, width=110, indent="    "):
    return textwrap.fill(str(s), width, initial_indent=indent, subsequent_indent=indent)

## 1. Inventory

**Why:** before any analysis, know the size of each test set and of each slice we would report on. A score on 20
questions means nothing, and a filter that drops half a file tells us that file is fragile.

**Look for:** how many rows survive the checks (`ok`, `answer_ok` in `rag`), whether dev (20%) is big enough per
language to tune on, and which question types dominate. `clean` and `challenge` are the source items (provenance),
not test sets: `rag` was built from their kept items.

In [ ]:
q("""
select 'rag' file, count(*) n, avg(answer_ok::int) ok, avg((split = 'dev')::int) dev from rag
union all select 'unanswerable', count(*), avg(ok::int), avg((split = 'dev')::int) from unanswerable
union all select 'compare', count(*), avg(ok::int), avg((split = 'dev')::int) from compare
union all select 'meta', count(*), avg(ok::int), avg((split = 'dev')::int) from meta
union all select 'tables', count(*), avg(ok::int), avg((split = 'dev')::int) from tables
union all select 'clean (source, easy)', count(*), avg((labels->>'verdict' = 'keep')::int), null from clean
union all select 'challenge (source, hard)', count(*), avg(challenge_ok::int), null from challenge
""").round(3)

In [ ]:
by_lang = (
    q("select lang, kind, count(*) n from rag where answer_ok group by all")
    .pivot(index="lang", columns="kind", values="n")
    .fillna(0)
)
by_lang = by_lang.assign(total=by_lang.sum(axis=1)).sort_values("total")
by_lang[["corpus", "challenge"]].plot.barh(
    stacked=True,
    figsize=(10, 7),
    logx=True,
    title="answer_ok rag questions per language (log scale)",
)
plt.show()
by_lang.sort_values("total", ascending=False).astype(int).T

In [ ]:
q("select qtype, kind, count(*) n from rag where answer_ok group by all").pivot(
    index="qtype", columns="kind", values="n"
).sort_values("corpus", ascending=False)

**So what**

- `rag` is large (499k usable) and nearly all of it passed the checks (96%), so size is not a problem. `tables`
  is the exception: 166 usable rows, only enough for a smoke test.
- The language mix is extreme: Swedish 193k, German 80k, Finnish 54k, and Dutch 5.9k (1%). mt and tr have about
  20 questions and are not usable. For Dutch the dev split has 892 easy and 379 hard questions: enough to read and
  sanity-check, not enough for fine-grained statistics.
- Hard questions are 71% `identify` ("which castle ..."): the hard set tests *finding a place from a
  description*. Easy questions ask for facts (numbers, descriptions) about a named place.

**Take into account:** always report per language (or on `balanced`), with the number of questions next to each
score; leave out languages with < 100 questions. Report easy and hard apart: they measure different skills.

**Not shown yet:** questions per page (pages with 9 questions weigh 9 times as much in an average) and question
length per language.

## 2. The corpus: what can be found at all

**Why:** the index is the other half of every retrieval score. How hard it is to find a page depends on what the
other 133k pages look like: thousands of near-identical lake stubs make a lake hard to single out, and a short
page gives an embedding little to work with.

**Look for:** the language and category mix, bot-written pages (Swedish lake register), stubs (< 2,000 chars) and
missing metadata. Together they decide what "a random page" looks like, and how far it is from a tourist page.

In [ ]:
corpus = q("""
select in_language lang, count(*) pages, avg(machine_generated::int) bot, avg((char_count < 2000)::int) stub,
       median(char_count)::int median_chars, avg(language_ok::int) language_ok, avg((latitude is not null)::int) has_coords
from pages group by 1 order by 2 desc""")
corpus.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
q(
    "select category, count(*) n from pages, unnest(categories) t(category) group by 1 order by 2"
).plot.barh(
    x="category",
    y="n",
    ax=ax[0],
    legend=False,
    title="pages per category (a page can have several)",
)
q("select country, count(*) n from pages group by 1 order by 2 desc limit 27").plot.bar(
    x="country",
    y="n",
    ax=ax[1],
    legend=False,
    logy=True,
    title="pages per country (log)",
)
plt.show()

In [ ]:
# category x language: where does each type of place come from?
cat_lang = (
    q(
        "select category, in_language lang, count(*) n from pages, unnest(categories) t(category) group by all"
    )
    .pivot(index="category", columns="lang", values="n")
    .fillna(0)
    .astype(int)
)
cat_lang[cat_lang.sum().sort_values(ascending=False).index[:12]]

**So what**

- Half the corpus is Swedish (69k pages), 81% of that bot-written (lake register) with a median of 1,659 chars.
  The typical page is a short, templated description of a Swedish lake, not a tourist attraction.
- Category and language are tied together: lakes are Swedish and Finnish, castles German, French and Spanish, and
  mountains German, Italian and Slovak. A "castle" result is mostly a German / French / Spanish result.
- Lithuanian, Estonian, Latvian and Irish pages are 80-95% stubs: little text to embed or chunk.
- Metadata is complete (every page has coordinates and a checked language), so filters on country, category and
  distance are possible.

**Take into account:** a chunking or embedding choice that helps long castle pages may do nothing for 500-char lake
stubs, and the other way round. Compare methods per category or per page length, not only overall. Templated
pages (`page_template_sim` close to 1) are where retrieval will confuse places.

**Not shown yet:** how far the corpus is from what tourists visit. Wikipedia page views (free Wikimedia API) would
give a visitor-interest signal next to `sitelinks` (notability) and `char_count` (documentation).

## 3. Popularity: did the generator favour famous places?

**Why:** this decides how far every score can be trusted. Generated datasets carry the biases of the models that
made them (SynthTRIPs found its generator favoured popular places even when asked for obscure ones). Famous places
are also the ones an LLM already knows and the ones with long, rich pages. If they get most questions, scores look
better than they will be on the long tail of small castles and lakes that a recommender should surface.

**Look for:** `sitelinks` (in how many Wikipedia languages a place has an article) is our notability signal. Does
every page get questions equally, or do famous pages get more? Short pages get fewer items by design (1-2 for
stubs), and Swedish lake stubs are half the corpus, so split by stub and by Swedish / other.

In [ ]:
pop = q("""
with qs as (select id, count(*) nq from rag where answer_ok group by 1)
select in_language = 'sv' swedish,
       case when sitelinks = 1 then '1' when sitelinks <= 3 then '2-3' when sitelinks <= 10 then '4-10'
            when sitelinks <= 30 then '11-30' else '31+' end sitelinks,
       char_count < 2000 stub, count(*) pages, avg((nq is not null)::int) has_question, avg(coalesce(nq, 0)) questions_per_page
from pages left join qs using (id) group by all order by swedish, min(pages.sitelinks), stub""")
pop.pivot_table(
    index=["sitelinks", "stub"],
    columns="swedish",
    values=["pages", "has_question", "questions_per_page"],
    sort=False,
).round(2)

In [ ]:
# share of pages vs share of questions per sitelinks bucket; questions above pages = over-represented
share = pop.groupby("sitelinks", sort=False).apply(
    lambda d: pd.Series(
        {"pages": d.pages.sum(), "questions": (d.pages * d.questions_per_page).sum()}
    )
)
(share / share.sum()).plot.bar(
    title="share of pages vs share of questions, by sitelinks", rot=0
)
plt.show()

In [ ]:
# the tourist_appeal tag (Gemma student) per kind: hard questions need pages >= 1,500 chars
q(
    """select kind, page_tags->>'tourist_appeal' tourist_appeal, count(*) n from rag where answer_ok group by all"""
).pivot(index="tourist_appeal", columns="kind", values="n")

**So what**

- No bias towards famous places in which pages get questions: outside Swedish, 80-97% of pages in every
  `sitelinks` bucket have questions, and famous pages do not get more per page (about 5.5).
- The one hole: Swedish lake stubs with a single language (31k pages) got a question on only 29% of pages. The
  generator skipped pages with almost nothing to ask about. That is sensible, but it means the obscure end of
  Swedish is under-sampled.
- Most questions are about places the page tagger rates low on tourist appeal (62% of hard questions). The set
  tests *finding obscure places*, not popular attractions: good for robustness, but not a sample of what tourists
  ask.

**Take into account:** coverage is not skewed by popularity, but *answering* may still be. An LLM can answer
questions about famous places from memory, so retrieval looks less necessary there. Break answer scores down by
`sitelinks` bucket.

Whether famous places are also *easier to find* is in section 4 (they are). **Not shown yet:** the closed-book test
(ask the local LLM without retrieval), split by `sitelinks`, to measure how much it already knows.

## 4. Difficulty: how much is solved by keyword matching?

**Why:** we are here to compare indexing methods (chunking, embeddings). If most questions can be solved by
matching the place name, every method scores ~100% and the comparison tells us nothing. We need to know which
slice has room for improvement, and what makes a question hard, so we can report on the slice that matters.

**Look for:** `rank_o` = gold page rank for the question: [dense rank, dense margin, BM25 rank], 101 = not in the
top 100. BM25 (keyword search) is the baseline to beat. `title_in_question` and `lex_overlap` (share of the
question's keywords found in the page) should explain most of BM25's success. Dense ranks only exist for about a
third of the rows, so check *which* rows before trusting them.

In [ ]:
diff = q("""
select kind, count(*) n, avg(title_in_question::int) title_in_q, avg(lex_overlap) lex_overlap,
       avg((rank_o[3] <= 1)::int) bm25_hit1, avg((rank_o[3] <= 10)::int) bm25_hit10,
       avg((rank_o[1] is not null)::int) has_dense, avg((rank_o[1] <= 10)::int) dense_hit10
from rag where answer_ok group by 1""")
diff.round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
lex = q(
    "select kind, lex_overlap from rag where answer_ok using sample 100000 (reservoir, 1)"
)
for kind, d in lex.groupby("kind"):
    d.lex_overlap.plot.hist(bins=40, alpha=0.5, ax=ax[0], label=kind)
ax[0].set(
    title="lex_overlap: share of the question's keywords in the page",
    xlabel="lex_overlap",
)
ax[0].legend()
per_lang = q(
    """select lang, avg((rank_o[3] <= 10)::int) bm25_hit10 from rag where answer_ok and kind = 'challenge' group by 1 having count(*) > 200 order by 2"""
)
per_lang.plot.barh(
    x="lang",
    y="bm25_hit10",
    ax=ax[1],
    legend=False,
    title="hard questions: BM25 hit@10 per language",
)
plt.tight_layout()
plt.show()

In [ ]:
# BM25 hit@10 by lex_overlap bucket: is lex_overlap a good difficulty knob?
q("""select kind, round(lex_overlap, 1) lex_overlap, count(*) n, avg((rank_o[3] <= 10)::int) bm25_hit10
from rag where answer_ok group by all order by all""").pivot(
    index="lex_overlap", columns="kind", values="bm25_hit10"
).round(2)

In [ ]:
# the headline number depends on the subset: all rows vs balanced (<= 400 pages per language)
q("""select kind, balanced, count(*) n, avg((rank_o[3] <= 1)::int) bm25_hit1, avg((rank_o[3] <= 10)::int) bm25_hit10,
       avg((hits = 'unique')::int) unique_ from rag where answer_ok group by all order by all""").round(
    3
)

In [ ]:
# which rows have a dense rank? If they are not a random sample, the dense column says little about the whole set
q("""select kind, rank_o[1] is not null has_dense, lang = 'sv' swedish, count(*) n,
       avg((rank_o[3] <= 10)::int) bm25_hit10, avg((rank_o[1] <= 10)::int) dense_hit10
from rag where answer_ok group by all order by all""").round(3)

In [ ]:
# are questions about famous places easier to find? (hard questions, outside Swedish)
q("""select case when sitelinks = 1 then '1' when sitelinks <= 3 then '2-3' when sitelinks <= 10 then '4-10' else '11+' end sitelinks,
       count(*) n, avg(lex_overlap) lex_overlap, avg((rank_o[3] <= 10)::int) bm25_hit10, avg((hits = 'unique')::int) unique_
from rag where answer_ok and kind = 'challenge' and lang <> 'sv' group by 1 order by min(sitelinks)""").round(
    3
)

In [ ]:
# where in the page is the answer? Matters for chunking: an index of the lead only misses deep evidence
q("""select kind, median(char_count) median_page_chars, avg((spans[1][1] > 1000)::int) first_evidence_after_1000_chars,
       avg((spans[1][1] > 2000)::int) after_2000_chars, avg(evidence_in_table::int) evidence_in_table
from rag where answer_ok and len(spans) > 0 group by 1""").round(3)

**So what**

- Easy questions are solved by keyword search: 82% name the place, BM25 hit@10 is 97%. They are a sanity check
  (is the index broken?), not a way to rank methods.
- Hard questions leave real room: never named, BM25 hit@1 40% and hit@10 66%. **This is the slice to compare
  methods on.** On `balanced` the same number is 81%: Swedish pulls the overall number down, so always say which
  subset a number is on.
- `lex_overlap` is a clean difficulty knob: BM25 hit@10 rises from ~0 at overlap 0.1 to ~80% at 0.8. Use it to
  stratify ("how does a method do on questions that share few words with the page?").
- The dense ranks in the file cannot be used: for hard questions they exist mostly for Swedish rows (95%), and the
  dense index saw only the title plus the first 1,000 chars per page. Outside Swedish only 3k hard questions have
  one, and on those the dense baseline (30% hit@10) is far below BM25 (90%). It measures a weak setup on a
  non-random subset, not "dense retrieval".
- Obscure places are harder: for hard questions outside Swedish, BM25 hit@10 is 56% for places with one language
  version against 82-87% for better-known ones, and they are more often ambiguous. An average over all questions
  hides how a method does on the long tail.
- About half the answers sit after the first 1,000 chars of the page, and 34% of the hard questions rely on a
  table row (mostly the infobox). That is why the lead-only dense index fails, and it is the core case for
  chunking: chunks must reach deep text, and tables must survive chunking.

**Take into account:** a difference between two methods on easy questions is noise. Report hard questions by
language, by `lex_overlap` bucket and by `sitelinks` bucket. Compute our own dense baseline over full pages or
chunks; do not reuse `rank_o[1]`. Check how the chunker handles Markdown tables.

**Caveat on "hard":** even hard questions keep 72% of their weighted words in the page (`lex_overlap`). The generator
read the page and reused its facts, numbers and place names; only the title and the 25 rarest words were banned.
Real users have not read the page, so real-world retrieval is likely harder than any score here. Section 13
measures the gap with your own questions.

## 5. Ambiguity: does a question single out one page?

**Why:** hit@k assumes one right page. If a question fits three lakes equally well, a system that returns another
fitting lake is scored wrong while it is right. Ambiguous questions make scores too low, and they make method
comparisons noisy.

**Look for:** for judged rows (`qrels_judged`, mostly hard questions), every page in the BM25 top 20 was judged by
an LLM. `hits` = unique / few / many; `relevant` lists the other fitting pages; `pool_saturated` = probably more
matching pages than judged. Expect Swedish lakes to stand out. Note that only BM25 candidates were judged: pages
only a dense model would find are unjudged.

In [ ]:
amb = q("""select lang, count(*) n, avg((hits = 'unique')::int) unique_, avg((hits = 'few')::int) few, avg((hits = 'many')::int) many,
       avg(pool_saturated::int) saturated, avg((gold_match = 'yes')::int) gold_yes
from rag where answer_ok and qrels_judged and kind = 'challenge' group by 1 having count(*) > 200 order by unique_""")
amb.round(3)

In [ ]:
q(
    """select coalesce(hits, 'not judged') hits, kind, machine_generated, count(*) n from rag where answer_ok group by all"""
).pivot_table(index="hits", columns=["kind", "machine_generated"], values="n")

**So what**

- Outside Swedish and Finnish, ~95-99% of hard questions single out one page: the generator did what it was asked.
- Swedish hard questions are 77% unique, and the ambiguous ones are almost all about bot-written lake pages
  (21k few / many). "Which small lake in Jämtland ..." often has no single right answer.
- Easy questions were mostly not judged (only balanced pages), so we do not know how ambiguous they are.
- The gold page itself is judged a match 92-99% of the time: a few percent of gold labels are doubtful even to
  the judge.

**Take into account:** score with `relevant` counted as correct (the README says to). Otherwise Swedish scores
are too low by design. Or report on `hits = 'unique'` only. Because the pool is BM25 only, a dense method may find
fitting pages nobody judged and still be scored wrong. Check a few of its "misses" by hand.

**Not shown yet:** how much scores change with and without `relevant` counted, per language. Do that as soon as
there is a baseline.

## 6. What the filters removed

**Why:** what a filter removes shapes what is left. If one check drops most rows, the set reflects that check's (and
that model's) idea of a good question. Rejected rows also show what the generator gets wrong, which tells us what to
look for in the kept ones.

**Look for:** which `answer_ok` criterion fails most (AND over the judges), how many rows were never judged, and
how strict the source filters were for easy (`clean`) and hard (`challenge`) items.

In [ ]:
crit = [
    "context_relevant",
    "answerable",
    "answer_relevant",
    "answer_supported",
    "answer_complete",
    "language_match",
    "answer_fluent",
]
fails = q(
    "select "
    + ", ".join(f"sum(((criteria->>'{c}') = 'false')::int) {c}" for c in crit)
    + ", count(*) n, sum((criteria is null)::int) not_judged from rag where not answer_ok or answer_ok is null"
)
fails.T.rename(columns={0: "rows failing"})

In [ ]:
# source items: verdicts of the easy items by generator, and the hard-question filters
display(
    q(
        "select source, labels->>'verdict' verdict, count(*) n from clean group by all"
    ).pivot(index="source", columns="verdict", values="n")
)
q(
    """select prompt, count(*) n, avg(challenge_ok::int) challenge_ok, avg(likely_ambiguous::int) likely_ambiguous from challenge group by 1 order by 1"""
)

**So what**

- Only 4% of `rag` failed, and half of the failures are `answer_supported`: the judge saw only the title plus the
  quoted evidence, not the page. In the Dutch examples (section 11, Eukenberg) the answer is right but one detail
  sits outside the quote. Some rejections are false alarms; the hand check (section 12) tells how many.
- About a third of the non-ok rows were simply never judged (`criteria` null), not rejected.
- The source filters were mild for easy items (98% keep) and stricter for hard ones (83% `challenge_ok`). Nearly
  all hard items come from prompt v3.

**Take into account:** `answer_ok = false` does not mean "wrong answer". Do not use those rows as negatives. The
kept set is shaped by two free preview models (Ling, Bunny) judging each other's output, checked by Sonnet on
small samples only. That is why the hand check (section 12) matters.

**Not shown yet:** whether the filters dropped some languages or categories more than others (a filter bias), for
example `answer_ok` rate per language and category.

## 7. Cross-lingual

**Why:** LLMs4EU is about all EU languages, and a tourist asks in their own language about a place described in
another. Cross-lingual retrieval is where keyword search cannot help and where multilingual embeddings must earn
their place. It is also the only large source of Dutch and English questions in this set.

**Look for:** every `rag` question has a translation into one other EU language (`question_x`, `x_lang`), spread
evenly by hash. `x_ok` needs both judges, and the report calls translations the weakest field. Compare retrieval
with the original and with the translated question.

In [ ]:
xl = q(
    "select lang, x_lang, count(*) n, avg(x_ok::int) x_ok from rag where answer_ok group by all"
)
fig, ax = plt.subplots(1, 2, figsize=(15, 6))
m = xl.pivot(index="lang", columns="x_lang", values="n").fillna(0)
ax[0].imshow(m, cmap="Blues", norm="log")
ax[0].set(
    xticks=range(len(m.columns)),
    xticklabels=m.columns,
    yticks=range(len(m.index)),
    yticklabels=m.index,
    title="question language -> translation language (log count)",
    xlabel="x_lang",
    ylabel="lang",
)
xl.groupby("x_lang").apply(
    lambda d: (d.n * d.x_ok).sum() / d.n.sum()
).sort_values().plot.barh(ax=ax[1], title="x_ok rate by translation language")
plt.tight_layout()
plt.show()

In [ ]:
# retrieval with the translated question: BM25 is monolingual, so expect a collapse
q("""select kind, avg((rank_o[3] <= 10)::int) bm25_hit10_original, avg((rank_x[3] <= 10)::int) bm25_hit10_translated,
       avg((rank_o[1] <= 10)::int) dense_hit10_original, avg((rank_x[1] <= 10)::int) dense_hit10_translated
from rag where answer_ok and x_ok group by 1""").round(3)

**So what**

- Translations cover every language pair fairly evenly, so cross-lingual results can be reported per question
  language *and* per page language.
- Translation quality varies: 97% `x_ok` into English, 86% into Dutch, lower for small languages. The report
  found errors in ~1 in 7 translations that *passed*. Read the Dutch ones (section 11) before trusting this slice.
- BM25 collapses on translated questions (hard: 62% -> 0.6% hit@10). Even the weak dense baseline drops less on
  easy questions (96% -> 90%). This slice clearly separates monolingual from multilingual methods.

**Take into account:** use only `x_ok` rows, and report by `x_lang`. A Dutch translation of a Swedish lake question
does not show how Dutch speakers phrase questions: these are translations, not native queries.

**Not shown yet:** a multilingual dense baseline on translated questions, which is the number we actually need.

## 8. The other files

**Why:** the end goal is a system that helps people find attractions, which needs more than one-page lookup. It
must say "I don't know" (unanswerable), combine pages (compare), return sets of places (meta) and, near borders,
cover both sides. These files show how far the set already tests that, and what is missing before the credits run
out on 10 October.

**Look for:** size and pass rate per file, and for geo questions whether the gold places cross borders and
languages.

In [ ]:
display(
    q(
        "select type, count(*) n, avg(ok::int) ok, avg(qrels_judged::int) judged from unanswerable group by 1"
    )
)
display(
    q(
        "select qtype, count(*) n, avg(ok::int) ok from compare group by 1 order by 2 desc"
    )
)
q(
    "select op, answer_source, count(*) n, sum(ok::int) ok from tables group by all order by 3 desc"
)

In [ ]:
meta = q("""
with g as (select key, unnest(gold_pages) gid from meta where ok)
select m.kind, m.radius_km, count(distinct m.key) questions, avg(m.n_gold) avg_gold,
       avg((langs > 1)::int) multi_language, avg((countries > 1)::int) multi_country
from meta m join (select key, count(distinct split_part(gid, '/', 1)) langs, count(distinct p.country) countries
                  from g join pages p on p.id = g.gid group by 1) x using (key)
where m.ok group by all order by all""")
meta.round(3)

In [ ]:
q("select n_gold, kind, count(*) n from meta where ok group by all").pivot(
    index="n_gold", columns="kind", values="n"
).plot.bar(title="gold places per meta question", rot=0)
plt.show()

**So what**

- Unanswerable (29k ok) is a solid set: half false premise, half not covered. Nearly all were checked against
  other pages too, so "no page answers it" was actually tested.
- Compare (8.5k) is mostly numbers and dates ("which is higher / older"): a test of reading two pages, not of
  recommending.
- Meta is almost all geo (3.8k) with 1-15 gold places; only 100 list questions. Geo questions cross a border in
  only 2-7% of cases, and a language in ~5%. Lists are per country by design.
- Tables (166 ok) is too small to report on.

**Take into account:** none of these are recommendation questions ("a quiet castle near X, good with kids"). Meta
questions are exhaustive lists with a gold set from metadata, so the gold list is only as complete as Wikidata.
List scores need set metrics (recall, precision), not hit@k.

**Missing, and worth generating before 10 October:** preference-style recommendation questions with a set of
acceptable answers, and cross-border geo questions (anchors near a border, gold pages in two languages). Both were
the top candidates in the audit plan, and these numbers confirm the gap.

## 9. Generator artifacts

**Why:** generated questions carry the fingerprints of the model and prompt that wrote them: favourite openings,
fixed lengths, recurring templates. Two risks follow. A retriever or LLM (especially one we fine-tune) can learn the
style instead of the task, and if one generator wrote most questions, our results describe that generator's style
rather than how people ask.

**Look for:** who wrote the hard questions (Bunny or Ling, by the report's hash rule), how much the openings repeat
per language, exact duplicates across pages, and templates in the unanswerable questions.

In [ ]:
# hard questions per generator, Swedish vs other; and how many of each generator's items survived the filters
display(
    q("""select lang = 'sv' swedish, generator(id) generator, count(*) n, avg(len(string_split(question, ' '))) words,
       avg((rank_o[3] <= 10)::int) bm25_hit10, avg((hits = 'unique')::int) unique_
from rag where answer_ok and kind = 'challenge' group by all order by all""").round(3)
)
q(
    "select generator(id) generator, count(*) written, avg(challenge_ok::int) kept from challenge group by 1"
).round(3)

In [ ]:
# how repetitive are the openings? share of questions starting with the language's 5 most common first two words
q("""with o as (select lang, kind, lower(array_to_string(string_split(question, ' ')[1:2], ' ')) opening from rag where answer_ok),
c as (select lang, kind, opening, count(*) n, row_number() over (partition by lang, kind order by count(*) desc) r
      from o group by lang, kind, opening)
select lang, kind, sum(n) total, round(sum(n) filter (where r <= 5) / sum(n), 2) top5_share,
       string_agg(opening, ' | ' order by r) filter (where r <= 3) top3
from c group by lang, kind having sum(n) > 1000 order by kind, top5_share desc""")

In [ ]:
# the same question asked about several pages: it can never single out one
q(
    "select question, count(distinct id) pages from rag where answer_ok group by 1 having count(distinct id) > 1 order by 2 desc"
)

In [ ]:
# unanswerable templates: the first three English words per type
q("""with o as (select type, lower(array_to_string(string_split(question_en, ' ')[1:3], ' ')) opening, count(*) n
           from unanswerable where ok group by type, opening)
select type, opening, n, round(n / sum(n) over (partition by type), 3) of_type from o
qualify row_number() over (partition by type order by n desc) <= 6 order by type, n desc""")

**So what**

- The report describes the hard questions as split evenly between two generators by hash, but outside Swedish 91%
  of the kept ones come from Ling. Bunny generated fewer (71k vs 214k, probably because of its 50 requests/min limit) and
  fewer of its questions survived (76% vs 85%). Outside Swedish, our hard set is in practice "the Ling hard set".
- Swedish questions are highly templated: 67% of hard and 72% of easy ones start with one of five openings
  ("vilken sjö", "hur stor"). Other languages range from 20% (hu, de) to 54% (ro) for hard questions. Lake stubs
  leave little else to ask, so the Swedish set mostly tests the same few question shapes.
- 31% of not-covered unanswerable questions ask "how much does ..." (ticket prices), and false-premise questions
  start with "why was / why is the ...". A system can score well on abstention by learning "price question -> I
  don't know" without reading the page.
- 11 questions are asked word for word about several pages (one about 5 Swedish lakes). The cross-page filter
  missed them.

**Take into account:** report hard-question results per generator as a robustness check. Score unanswerable per
template: a system that refuses every price question is not abstaining well. Never fine-tune on these questions
and test on them too: the templates leak.

**Not shown yet:** the easy questions' generator (Qwen LoRA vs Bunny teacher) is not in the `rag` file, only in
`clean.source`. Joining it in would allow the same check for easy questions.

## 10. Data hygiene

**Why:** three cheap checks that change how we prepare and score the data. (1) Did the answer filter remove some
languages or page types more than others? Then the kept set is shifted. (2) How much of the page text is wiki
boilerplate? Every chunk and embedding sees it. (3) Which answers can change over time? Those go stale in a fixed
test set.

In [ ]:
# (1) filter bias: share of generated rows that passed answer_ok, per language and per category
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
q(
    "select lang, avg(coalesce(answer_ok, false)::int) ok from rag group by 1 having count(*) > 300 order by 2"
).plot.barh(
    x="lang",
    y="ok",
    ax=ax[0],
    legend=False,
    xlim=(0.75, 1),
    title="answer_ok rate per language",
)
q(
    "select category, avg(coalesce(answer_ok, false)::int) ok from rag, unnest(categories) t(category) group by 1 order by 2"
).plot.barh(
    x="category",
    y="ok",
    ax=ax[1],
    legend=False,
    xlim=(0.75, 1),
    title="answer_ok rate per category",
)
plt.tight_layout()
plt.show()

In [ ]:
# (2) boilerplate in the page text: edit links like [bewerken | brontekst bewerken], footnotes [1], table lines
q(rf"""select in_language lang, count(*) pages,
       avg((regexp_matches(text, '\[[^\]\n]{{2,40}}\|[^\]\n]{{2,40}}\]'))::int) pages_with_edit_links,
       1 - sum(length(regexp_replace(text, '\[[^\]\n]{{2,40}}\|[^\]\n]{{2,40}}\]', '', 'g'))) / sum(length(text)) edit_link_chars,
       1 - sum(length(regexp_replace(text, '\[\d{{1,3}}\]', '', 'g'))) / sum(length(text)) footnote_chars,
       1 - sum(length(regexp_replace(text, '(?m)^\|.*$', '', 'g'))) / sum(length(text)) table_chars
from read_json_auto('{PAGES}') group by 1 having count(*) > 200 order by 2 desc""").round(
    3
)

In [ ]:
# (3) answers that may change (Ling tag; the README says it over-flags ~1 in 8)
q(
    "select qtype, count(*) n, avg(time_sensitive::int) time_sensitive from rag where answer_ok group by 1 order by 3 desc"
).round(3)

**So what**

- The answer filter was harder on rich pages: `answer_ok` is 81-84% for Hungarian and Greek and 88-90% for
  castles, national parks and gardens, against 95% for lakes. Longer, richer answers fail `answer_supported` more
  often, so the kept set leans slightly towards simple pages. It is not large, but it points the same way as the
  Swedish skew.
- Wikipedia boilerplate is everywhere: edit links (`[redigera | redigera wikitext]`, `[Bearbeiten | Quelltext
  bearbeiten]`, ...) are on ~95-100% of pages in most languages and make up 3.6% of all characters. Tables
  (infoboxes, data tables) are 22% of all characters, 39% for Swedish. Spanish, Croatian and English pages have no
  edit links, so the scraping differed per wiki.
- 20% of `description` answers and 13% of `name` answers may change over time; numbers and locations hardly.

**Take into account:** strip edit links and footnote markers before chunking and embedding; they are identical
tokens in every chunk and pull embeddings together. Decide deliberately how tables are chunked (kept whole, turned
into sentences, or dropped): they hold a fifth of the text and a third of the hard-question evidence. Grade
`time_sensitive` answers leniently or leave them out of strict answer scoring.

## 11. Read examples

**Why:** statistics show what the labels *say*; reading shows whether the labels are *right*. All labels come from
LLMs, so read a few of each kind in a language you know. Dutch is our advantage here: we can judge native questions,
answers and translations that most of the team cannot.

**How:** set `LANG` and `SEED` in the first cell. Every example shows the question, its English gloss, the answer
and the evidence where it sits in the page (`⟦…⟧`).

**Check per example:** would a person ask this? Does it single out the place? Is the answer right and supported by
the page? For translations: is the Dutch natural and faithful? For rejections: was the judge right?

In [ ]:
def show_rag(df, context=250):
    page = texts(df.id)
    for r in df.itertuples():
        print(
            f"── {r.id}  {r.title}  ({r.kind}, {r.qtype}, hits={r.hits}, bm25 rank={r.bm25_rank}, answer_ok={r.answer_ok})"
        )
        print(f"  Q  {r.question}")
        if r.q_en and r.q_en != r.question:
            print(f"  EN {r.q_en}")
        print(f"  A  {r.answer}")
        if r.x_lang in ("en", "nl"):
            print(f"  {r.x_lang.upper()}→ {r.question_x}")
        text = page.get(r.id, "")
        for s, e in list(r.spans)[:2]:
            print(
                wrap(
                    "…"
                    + text[max(0, s - context) : s]
                    + "⟦"
                    + text[s:e]
                    + "⟧"
                    + text[e : e + context]
                    + "…"
                )
            )
        print()


def rag_sample(where, n=5):
    return q(f"""select r.*, g.q_en, rank_o[3]::int bm25_rank from rag r left join gloss g on g.id = r.id and g.q = r.question
               where {where} order by hash(r.id || r.n || {SEED}) limit {n}""")

In [ ]:
# easy questions: name the place
show_rag(rag_sample(f"answer_ok and lang = '{LANG}' and kind = 'corpus'"))

In [ ]:
# hard questions: describe the place without naming it
show_rag(rag_sample(f"answer_ok and lang = '{LANG}' and kind = 'challenge'"))

In [ ]:
# hard questions BM25 misses (rank > 10): really hard, or ambiguous?
show_rag(
    rag_sample(
        f"answer_ok and lang = '{LANG}' and kind = 'challenge' and rank_o[3] > 10", 4
    )
)

In [ ]:
# questions that fit several pages: are the other pages really fitting?
df = rag_sample(f"answer_ok and lang = '{LANG}' and hits in ('few', 'many')", 3)
show_rag(df)
for r in df.itertuples():
    print(r.id, "relevant:", list(r.relevant or [])[:8])

In [ ]:
# rows the answer judges rejected: which criterion failed, and was the judge right?
df = rag_sample(f"answer_ok = false and lang = '{LANG}'", 4)
for c, r in zip(df.criteria, df.itertuples()):
    print(r.id, "failed:", [k for k, v in json.loads(c).items() if v is False])
show_rag(df)

In [ ]:
# other languages, read through a Dutch or English translation: translation quality in a language we can judge
show_rag(rag_sample(f"answer_ok and x_lang = '{LANG}' and lang not in ('en', 'nl')", 5))

In [ ]:
# query variants (balanced pages): keyword, typo, verbose, history, followup
for r in q(
    f"select question, variants from rag where variants is not null and lang = '{LANG}' order by hash(id || n || {SEED}) limit 3"
).itertuples():
    v = json.loads(r.variants)
    print("Q", r.question)
    for k in ["keyword", "typo", "verbose", "history", "followup"]:
        print(
            f"  {k:9}",
            v.get(k),
            "" if v["check"].get(f"{k}_ok", True) else "  ✗ failed check",
        )
    print()

In [ ]:
# unanswerable: is the premise really false / the fact really absent?
for r in q(
    f"select * from unanswerable where ok and lang = '{LANG}' order by hash(id || type || {SEED}) limit 6"
).itertuples():
    print(
        f"── {r.id}  {r.title}  ({r.type})\n  Q  {r.question}\n  EN {r.question_en}\n  why: {r.why}\n"
    )

In [ ]:
# compare: needs both pages
for r in q(
    f"select * from compare where ok and lang = '{LANG}' order by hash(id || {SEED}) limit 4"
).itertuples():
    print(
        f"── {r.pages[0]} vs {r.pages[1]}  ({r.qtype})\n  Q  {r.question}\n  EN {r.question_en}\n  A  {r.answer}"
    )
    print(
        "  evidence A:",
        list(r.evidence_a)[:2],
        "\n  evidence B:",
        list(r.evidence_b)[:2],
        "\n",
    )

In [ ]:
# meta: is the gold list complete and correct? Think of places you know that are missing.
df = q(
    f"select * from meta where ok and lang = '{LANG}' order by hash(key || {SEED}) limit 4"
)
titles = dict(
    con.execute(
        "select id, title from pages where list_contains(?, id)",
        [[g for gp in df.gold_pages for g in gp]],
    ).fetchall()
)
for r in df.itertuples():
    print(
        f"── {r.key}  ({r.kind}, {r.n_gold} gold)\n  Q  {r.question}\n  EN {r.question_en}"
    )
    print(
        wrap(
            ", ".join(
                f"{titles.get(g, g)} [{g.split('/')[0][:2]}]" for g in r.gold_pages
            )
        ),
        "\n",
    )

In [ ]:
# tables: the answer is computed by code; check it against the page table
for r in q(
    f"select * from tables where ok order by (in_language = '{LANG}') desc, hash(id || n || {SEED}) limit 3"
).itertuples():
    print(
        f"── {r.id}  {r.title}  ({r.op}, {r.answer_source})\n  Q  {r.question}\n  EN {r.question_en}\n  answer: {r.answer_final}\n  header: {list(r.table)}\n"
    )

**So what (seed 7, Dutch)**

- Easy and hard questions read as natural Dutch, and the answers match the page. Hard questions do describe the
  place without naming it (Kasteel de Binckhorst via "first mentioned in 1308").
- Some hard questions BM25 misses are contrived ("what lies between the woodland with the memorial and the two
  forests"): hard for retrieval, but not how people ask. Hard ≠ realistic.
- At least one rejected row (Eukenberg) was right: the judge missed a detail outside the quoted evidence.
- The page text contains Wikipedia noise: `[bewerken | brontekst bewerken]`, infobox tables as `| |||`, footnote
  markers `[1]`. Chunking and embedding will see this unless it is cleaned.
- Not-covered unanswerable questions repeat a template ("how deep is ...", "how high is ..."). "No page answers it"
  is the generator's assumption, checked only against the BM25 pool.
- In these examples the `keyword` variant failed its check on reasonable keyword queries: the variant checks may be strict.

**From the hand check so far** (21 Dutch items: 10 easy, 10 hard, 1 ambiguous; preliminary):

| check | easy | hard |
|---|---|---|
| answer correct and in the page | 8/10 (2 unsure) | 10/10 |
| points to one place | 9/9 | 9/10 |
| fluent Dutch | 8/9 | 5/9 |
| would a real person ask this | 9/9 | 4/10 |

No answer was judged wrong. The answers sometimes rely on a detail outside the highlighted evidence (Spuikom), so
open the full page before marking an answer unsupported. Notes from the reviewer on hard questions: *"if you know
this, you don't need to know the name unless it's a riddle"* and *"if you know this, you also know the name"*. One
easy question was time-sensitive (a fact "since 2020"). The Dutch questions, though, often read as
translated: stiff constructions, wrong tense, *de/het* errors. Hard questions are riddles that stack details from
the page, and the forbidden-word rule can bend facts: Doodemanskisten, a dune lake, became "een Fries
natuurgebied aan het Groene Strand" with two islands "in het midden" (of what is not said).

**Take into account:** a few examples are anecdotes. Change `SEED`, read 20-30, and put the verdicts in the
hand-check sample below so they become a number.

### Examples behind the findings, in English

The summary's points about realism are easier to see in examples. First the hand-checked Dutch items with their
English translation. "How a person might ask" is **written by us** to illustrate the difference; it is not data.
"Shared words" are the question's words (5+ letters) that also occur in the page: what keyword search matches on.

In [ ]:
EXHIBITS = [
    (
        "nlwiki/Q2550802",
        6,
        "riddle: stacks a detail only a reader of the page knows; still BM25 rank 1",
        "Welke meren liggen er ten zuiden van Groningen? / Which lakes are south of Groningen?",
    ),
    (
        "nlwiki/Q2171082",
        7,
        "riddle + stiff Dutch ('wordt ... gaf', 'een vertrouwde man' for 'vertrouweling')",
        "Welk kasteel in Den Haag komt uit de middeleeuwen? / Which castle in The Hague dates from the Middle Ages?",
    ),
    (
        "nlwiki/Q13135245",
        7,
        "banned words bend the facts: a dune lake becomes a 'natuurgebied', islands 'in the middle' of nothing; grammar error (een ...-eeuwse boswachtershuis die)",
        "Welk meertje op Terschelling heeft twee eilandjes? / Which small lake on Terschelling has two islands?",
    ),
    (
        "nlwiki/Q2614106",
        0,
        "overspecified: two locators (national park and village) to make it unique",
        "Hoe hoog is de Elsberg? / How high is the Elsberg?",
    ),
    (
        "nlwiki/Q113636275",
        2,
        "stiff: 'over' twice",
        "Hoe zag kasteel Rhijnestein in Deil eruit? / What did Rhijnestein castle in Deil look like?",
    ),
    (
        "nlwiki/Q136684929",
        2,
        "fine question, but the highlighted evidence misses the 1912 sentence the answer relies on: an evidence-only judge would reject a correct answer",
        "-",
    ),
]


def shared_words(question, text):
    page_words = set(re.findall(r"\w{5,}", text.lower()))
    return sorted(
        {w for w in re.findall(r"\w{5,}", question.lower()) if w in page_words}
    )


rows = pd.concat(
    [
        q(f"""select r.id, r.n, r.title, r.question, g.q_en, r.answer, r.lex_overlap, rank_o[3]::int bm25_rank
                   from rag r left join gloss g on g.id = r.id and g.q = r.question where r.id = '{i}' and r.n = {n}""")
        for i, n, *_ in EXHIBITS
    ]
)
page = texts(rows.id)
for r, (_, _, problem, human) in zip(rows.itertuples(), EXHIBITS):
    print(f"── {r.title}  (BM25 rank {r.bm25_rank}, lex_overlap {r.lex_overlap:.2f})")
    print(f"  NL       {r.question}\n  EN       {r.q_en}\n  problem  {problem}")
    print(f"  shared   {', '.join(shared_words(r.question, page[r.id]))}")
    print(f"  a person might ask: {human}\n")

The same pattern in questions that were **written in English**, so no translation is involved: hard questions that
share most of their words with the page and are found at BM25 rank 1, longest first.

In [ ]:
en = q("""select id, title, question, answer, lex_overlap, rank_o[3]::int bm25_rank from rag
          where answer_ok and lang = 'en' and kind = 'challenge' and qtype = 'identify' and rank_o[3] = 1 and lex_overlap > 0.8
          order by len(question) desc limit 4""")
page = texts(en.id)
for r in en.itertuples():
    print(
        f"── {r.title}  (BM25 rank {r.bm25_rank}, lex_overlap {r.lex_overlap:.2f})\n  Q  {r.question}"
    )
    print(f"  shared  {', '.join(shared_words(r.question, page[r.id]))}\n")

## 12. Hand check

**Why:** every label was made and checked by LLMs. Sonnet spot checks used samples of 30-200 across all
languages. Before we report a score, we need our own estimate of how often the gold label is wrong: a 2%
difference between methods means nothing if 10% of the labels are wrong.

**How:** run the review app and click through the sample (65 items, about an hour):

```
uv run --group notebook python notebooks/review_app.py    # opens on a free port; VS Code forwards it
```

It shows each question with its English gloss, the answer, and the gold page(s) with the evidence highlighted, and
asks yes / no / unsure per check: natural, fluent, singles out the place, answer correct; for translations,
faithful and natural; for rejected rows, whether the judge was right. Labels are saved to
`notebooks/out/review_labels.jsonl` as you click, so you can stop and resume. The sample is fixed in
`out/review_items.json` (mostly Dutch, some English). Delete that file for a new sample. The cell below turns the
labels into a rate per stratum. With ~10 items per stratum the margin is wide (±20-30%), so read it as "clearly
fine / clearly a problem", not as a precise error rate.

In [ ]:
labels_file = Path("out/review_labels.jsonl")
if labels_file.exists():
    labels = pd.read_json(labels_file, lines=True).drop_duplicates(
        ["key", "check"], keep="last"
    )
    votes = labels[labels.check != "notes"]
    display(
        votes.pivot_table(
            index=["lang", "stratum"],
            columns="check",
            values="value",
            aggfunc=lambda v: f"{(v == 'yes').mean():.0%} of {len(v)}",
        )
    )
    display(
        labels[(labels.check == "notes") & (labels.value != "")][
            ["stratum", "key", "value"]
        ]
    )
else:
    print("no labels yet: run notebooks/review_app.py")

### Your verdicts vs the dataset's own `realistic` label

The `realism` group in the review app (20 hard questions, half labelled realistic by the dataset's labeller, label not
shown) plus the hard questions you already judged. The four cells of the confusion matrix show whether the label is
too lenient (you: no, label: true) or too strict (you: yes, label: false). The `tourist` check asks the narrower question for this
project (would a tourist ask it?), and `clues` pilots "could someone know these clues without reading the page?".

In [ ]:
if labels_file.exists():
    hard = labels[
        labels.stratum.isin(["hard", "realism"])
        & labels.check.isin(["natural", "tourist", "clues"])
    ].copy()
    hard[["id", "n"]] = hard.key.str.split("|", expand=True)[[1, 2]]
    con.register(
        "hard_votes", hard.assign(n=hard.n.astype(int))[["id", "n", "check", "value"]]
    )
    votes = q(
        "select h.check as check_, h.value as you, r.labels->>'realistic' as dataset_label, r.title from hard_votes h join rag r using (id, n)"
    )
    for check in ["natural", "tourist", "clues"]:
        v = votes[votes.check_ == check]
        if len(v):
            print(f"{check}: your verdict (rows) vs label (columns), n = {len(v)}")
            display(pd.crosstab(v.you, v.dataset_label, margins=True))

## 13. Your own questions vs generated ones

**Why:** every question in this set was written by a model that had just read the page, so the questions reuse the
page's words. Real users have not read the page. The cleanest way to see how optimistic the scores are is a small
set of questions written *without* looking at the pages, scored with the same search as the generated ones.

**How:** write 20-30 questions in `notebooks/own_questions.csv` (columns `question,gold_id`). Pick places you know
vaguely (a Dutch castle, lake or nature area), ask the way you would ask a chatbot, and only then look up the page
id, e.g. `q("select id, title from pages where title ilike '%binckhorst%'")`. Mix named places and half-remembered
ones. The cell builds a BM25 index over all 133k pages (DuckDB full-text search, ~25 s) and ranks your questions
and a sample of generated hard questions in the same languages with it.

**Look for:** hit@1 / hit@10 / MRR and word overlap with the gold page, own vs generated. Expect lower overlap and
lower hit rates for your questions. The difference is how much the generated set overstates real performance.

In [ ]:
OWN = Path("own_questions.csv")
own = (
    pd.read_csv(OWN) if OWN.exists() else pd.DataFrame(columns=["question", "gold_id"])
)
if own.empty:
    print("no own questions yet: fill notebooks/own_questions.csv (question,gold_id)")
else:
    con.sql("install fts; load fts")
    con.sql(
        f"create or replace table fts_pages as select id, title || ' ' || text body from read_json_auto('{PAGES}')"
    )
    con.sql(
        "pragma create_fts_index('fts_pages', 'id', 'body', stemmer='none', stopwords='none', ignore='(\\.|[^a-z\\p{L}0-9])+', strip_accents=0, overwrite=1)"
    )

    def bm25_rank(question, gold):
        top = [
            r[0]
            for r in con.execute(
                "select id from (select id, fts_main_fts_pages.match_bm25(id, ?) s from fts_pages) where s is not null order by s desc limit 100",
                [question],
            ).fetchall()
        ]
        return top.index(gold) + 1 if gold in top else 101

    def overlap(question, text):
        # share of the question's words (4+ letters) that occur in the page
        words = set(re.findall(r"\w{4,}", question.lower()))
        return len(words & set(re.findall(r"\w{4,}", text.lower()))) / max(
            len(words), 1
        )

    langs = ", ".join(sorted({f"'{g.split('wiki/')[0]}'" for g in own.gold_id}))
    gen = q(
        f"select question, id gold_id from rag where answer_ok and kind = 'challenge' and lang in ({langs}) order by hash(id || n || {SEED}) limit {max(30, len(own))}"
    )
    both = pd.concat(
        [own.assign(source="own"), gen.assign(source="generated hard")],
        ignore_index=True,
    )
    page = texts(both.gold_id)
    both["bm25_rank"] = [bm25_rank(r.question, r.gold_id) for r in both.itertuples()]
    both["word_overlap"] = [
        overlap(r.question, page.get(r.gold_id, "")) for r in both.itertuples()
    ]
    display(
        both.groupby("source")
        .agg(
            n=("question", "size"),
            hit1=("bm25_rank", lambda r: (r <= 1).mean()),
            hit10=("bm25_rank", lambda r: (r <= 10).mean()),
            mrr=("bm25_rank", lambda r: (1 / r.where(r <= 100)).fillna(0).mean()),
            word_overlap=("word_overlap", "mean"),
        )
        .round(2)
    )
    display(
        both[both.source == "own"][["question", "gold_id", "bm25_rank", "word_overlap"]]
    )

## 14. What to measure

The audit turns into a measurement plan. In order of priority:

1. **Baseline on hard questions, per language:** BM25 and one multilingual dense model over chunks, scored hit@1,
   hit@10 and MRR with `relevant` counted as correct, on dev. Break it down by `lex_overlap` bucket and `sitelinks`
   bucket. This is the yardstick for every chunking and embedding choice.
2. **Chunking where the answer is deep:** the same, split by evidence position (after 1,000 chars or not) and by
   `evidence_in_table`, to see what chunking actually fixes.
3. **Cross-lingual:** the dense baseline on `question_x` (only `x_ok` rows), by question language.
4. **Label error rate:** the hand check from section 12, per stratum. Every reported difference must be larger
   than this.
5. **Closed-book:** the local LLM without retrieval, by `sitelinks`, to measure what it already knows.
6. **Unanswerable:** does the system say "I don't know"? Measure refusals on unanswerable questions *and* on
   answerable ones (refusing everything must not score well), per question template.
7. **Realism gap:** own questions vs generated ones (section 13). Repeat it for every method we compare: a method
   that wins on generated questions but not on human ones is fitting the generator, not the users.

What not to measure: easy questions overall (always ~97%), one number across languages, scores on tables, or the
dense ranks already in the file.

## 15. Generating more realistic questions

**The problem:** the generator writes a question *while reading the page*. So the questions (1) reuse the page's
words and facts, (2) stack details only a reader would know, and (3) for hard questions, turn into "which place is
this?" riddles. Banning the rarest words only partly helps, and it can bend the facts. Real users remember little,
use their own words, often know the name and want a fact, or want a suggestion with several good answers.

**Methods that weaken the link between page text and question**, roughly from cheapest to most work:

| method | how | what it fixes | watch out for |
|---|---|---|---|
| Human few-shot examples | write 30-50 real-style Dutch / English questions (section 13 gives a start); put them in the prompt as examples of tone and length | stiff, over-detailed style; length | style only, the content still comes from the page |
| Generate from another source than the indexed page | about half the places have an article in another language (`sitelinks` > 1) or Wikidata facts; write the question from *that* source, gold = our page, then check our page answers it | copied vocabulary: the words come from a different text, as with a real user | only places with a second source; must verify the indexed page really answers it |
| Two-step "forget the page" | step 1: the model reads the page and writes 2-3 vague impressions (type, region, one memorable thing); step 2: a separate call writes the question from the impressions only, without the page | stacked details and copying; produces half-remembered questions | more ambiguous questions: needs the multi-page relevance judging (`relevant`) on every question |
| Query-first, page-second | the model writes a plausible visitor need from the place type and region only ("a castle ruin to visit with kids in Limburg"); search the corpus; an LLM judges which pages satisfy it (pooled over BM25 + dense results) | recommendation-style questions with several right answers: the end goal, and missing today | gold = a judged set, not one page; the judging pool must include dense results, not only BM25 |
| Persona + preferences (as in SynthTRIPs) | combine a persona ("family with young kids, no car"), a region and preferences with page attributes (category, `page_tags`, coordinates); the gold set comes from metadata and is judged | realistic recommendation requests, and controlled popularity (ask about obscure places on purpose) | the generator still favours famous places: check gold `sitelinks` after generating |
| Overlap ceiling with rejection | generate as now, but reject questions above a `lex_overlap` limit or with BM25 rank 1 | removes the most copied questions | selects the vaguest, often ambiguous questions; does not make them more natural |

**Checked against a second opinion.** A separate (web) answer on avoiding lexical leakage recommended six things.
Mapped onto this dataset:

| recommendation | status here | worth it? |
|---|---|---|
| write the question from an extracted fact, not the page | not done | **yes**, the strongest idea; better still from Wikidata or another-language source, since extracted facts are still page sentences |
| simulate a real user; few-shots from real query logs (Natural Questions, MS MARCO) | prompt asked for "what a real person would ask" | partly: those logs are English only; for Dutch our own seed set is the best few-shot source |
| multi-hop questions that hide a bridge entity | not done (`compare` names both places) | low priority: a different skill, rarely asked by tourists; links are not in the corpus text |
| a rewrite pass that removes overlapping words | **tried and rejected**: Sonnet kept 13/30, the places lost their uniqueness | skip, unless every rewrite gets a uniqueness check |
| filter on IDF-weighted overlap and BM25 rank | exists: `lex_overlap`, `rank_o[3]`, plus the banned rare words | use to **stratify**, not to filter (see below) |
| answerability, uniqueness and closed-book checks | first two exist; the uniqueness pool is BM25 only; **no closed-book check** | yes: add a dense model to the uniqueness pool; run closed-book by `sitelinks` |

**Is BM25 rank a reliable leakage check?** As a signal, yes: here BM25 hit@10 rises smoothly from ~0% at
`lex_overlap` 0.1 to ~80% at 0.8. But:

- the numbers depend on the BM25 setup (tokenisation, stemming, one index or one per language); our DuckDB BM25 in
  section 13 gives different ranks than the dataset's. Compare questions within one setup; thresholds do not carry
  over.
- a bad rank does not mean a good question: ambiguity (Swedish lakes), wrong facts or garbled text also give bad
  ranks. "Hard" and "broken" look the same without the uniqueness and answerability checks.
- filtering on one retriever biases the set against it. It already happened here in a small way: hard questions
  where both BM25 and the (weak, lead-only) dense model missed the top 10 were dropped as "likely ambiguous"
  (~1% of hard items), but only 63% of those were ambiguous to Sonnet, so some truly hard questions were removed.
  Small, but towards optimistic.
- the most useful comparison is relative: overlap and BM25 rank of generated questions vs human-written ones
  (section 13), in the same setup.

**What to do before 10 October**, given limited credits: (1) write the human seed set (it also serves section 13),
(2) generate a few thousand **query-first / persona** recommendation questions for the balanced pages, judged over a
BM25 + dense pool, and (3) as a pilot, a few hundred **other-source** hard questions, to test whether they come
out less copied. Check every new set with the same tools as here: `lex_overlap`, BM25 rank vs the human set, and a
hand check of 30 items.

## Findings (fill in)

**Message for the dataset creator** (draft): *"The questions are well-formed and mostly correct, but they are
written by a model that just read the page: they reuse 72% of its weighted words, and most hard questions are
'which place is this?' riddles. That is fine for comparing methods against each other, but it probably overstates
performance on real user questions, which tend to be vaguer and use other words. I'm testing that with a small set
of my own questions."*

- Hand check, share judged correct per stratum (section 12): so far (n = 21, Dutch easy and hard) answers are
  right (no answer judged wrong); hard questions fail on realism (natural 4/10, fluent 5/9), easy ones do not
  (9/9, 8/9). Still to do: rejected (were the judges too strict?), translation, unanswerable, compare, meta, English.
- Surprises while reading:
- What to generate before 10 October: